# Arena competition — starter

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/convexpi/missions/blob/main/starters/arena_starter.ipynb)

This is the starter for the **Arena** competitions — you connect a **trading agent** to a live
limit-order book over a WebSocket and are ranked by PnL. Set `ARENA_URL` to the competition's
WebSocket URL (shown on the competition page), then run your agent.

## Setup

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q "git+https://github.com/convexpi/arena.git@fd32b8da5f302b9d0c53ce44505becc862312625" websockets

In [ ]:
import asyncio
from convexpi.arena.client import RemoteAgent, MarketState

# From the competition page (e.g. arena-open / arena-book / arena-l3). Use the wss:// URL.
ARENA_URL = "wss://arena-production-e3f1.up.railway.app"
AGENT_ID  = "my-agent"          # pick a unique name; it appears on the leaderboard
print("Configured for", ARENA_URL)

## A simple market-making agent

`on_tick(state)` is called every tick with the current `MarketState` (best bid/ask, your position,
PnL). Return a list of orders. This agent quotes a tight two-sided market and manages inventory —
a baseline to beat. Edit `decide()` to implement your own logic.

In [ ]:
def decide(state: MarketState):
    """Return a list of orders for this tick. Quote around the mid, skew against inventory."""
    bid, ask = state.best_bid, state.best_ask
    if bid is None or ask is None:
        return []
    mid = (bid + ask) / 2
    edge = max(1, int((ask - bid) * 0.25))      # quote inside the spread
    skew = -state.position // 5                  # lean against inventory
    return [
        {"side": "buy",  "price": int(mid - edge + skew), "qty": 1, "type": "limit"},
        {"side": "sell", "price": int(mid + edge + skew), "qty": 1, "type": "limit"},
    ]

async def main():
    agent = RemoteAgent(ARENA_URL, agent_id=AGENT_ID, on_tick=decide)
    await agent.run()

# In Colab/Jupyter:  await main()
# As a script:       asyncio.run(main())
print("Define your strategy in decide(), then run the agent.")

Watch your agent climb the live board on the competition page. The realistic exchanges
(*arena-book*, *arena-l3*) add real depth, slippage, and FIFO queue position — see Missions 2 and 7
for the microstructure that decides whether your quotes actually fill.